# 0 - Construction de la base de données

## Importation des modules

In [1]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import json
import pandas as pd
import sys
import yaml

# Ajout du chemin
sys.path.append('..')

# Importation des modules ad hoc
from dashboard_template_database.builders.schema import SchemaBuilder
from dashboard_template_database.builders.tables import DuckdbTablesBuilder
from dashboard_template_database.storage.loader import Loader

# Chargement du fichier de configurations
with open("../config.yaml") as file:
    config = yaml.safe_load(file)

# Chargement du fichier de parmaètres
with open("../parameters/labels.json") as file:
    labels = json.load(file)


## Importation des données

In [2]:
# Initialisation du loader
loader = Loader()
# Importation des données
df_origin = loader.load(filepath=os.path.join('../', config['INPUT_DATA']))
# Conversion en datetime
df_origin['date'] = pd.to_datetime(df_origin['date'])
df_origin.head()

,indicator,country,date,value,kind,horizon,week,model,training
0,Gross Domestic Product,France,1960-04-01,0.375710,observed,NaN,NaN,NaN,NaN
1,Gross Domestic Product,France,1960-07-01,0.748561,observed,NaN,NaN,NaN,NaN
2,Gross Domestic Product,France,1960-10-01,1.185218,observed,NaN,NaN,NaN,NaN
3,Gross Domestic Product,France,1961-01-01,1.608374,observed,NaN,NaN,NaN,NaN
4,Gross Domestic Product,France,1961-04-01,1.600329,observed,NaN,NaN,NaN,NaN


## 1 - Construction du schéma

### Initialisation de la classe

In [3]:
# Initialisation du schéma
schema_builder = SchemaBuilder(df=df_origin, categorical_threshold=config['THRESHOLD'])

### Construction des méta-données

In [4]:
# Construction du jeu de métadonnées
df_metadata = schema_builder.create_metadata_table(column_labels=labels)

df_metadata.head()

2026-01-22 20:13:36,330 - INFO - Successfully extracted meta-data from column 'indicator'
2026-01-22 20:13:36,394 - INFO - The column 'indicator' is of type 'object' and the number of modalities 2 satisfies the categorical threshold criteria 200
2026-01-22 20:13:36,396 - INFO - Successfully extracted meta-data from column 'country'
2026-01-22 20:13:36,436 - INFO - The column 'country' is of type 'object' and the number of modalities 6 satisfies the categorical threshold criteria 200
2026-01-22 20:13:36,438 - INFO - Successfully extracted meta-data from column 'date'
2026-01-22 20:13:36,441 - INFO - Successfully extracted meta-data from column 'value'
2026-01-22 20:13:36,442 - INFO - Successfully extracted meta-data from column 'kind'
2026-01-22 20:13:36,487 - INFO - The column 'kind' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 200
2026-01-22 20:13:36,489 - INFO - Successfully extracted meta-data from column 'horizon'
2026-01-22 20:13:

,name,label,python_type,sql_type,is_categorical,is_primary_key
0,country,Country,object,VARCHAR,True,False
1,date,Date,datetime64[ns],TIMESTAMP,False,False
2,horizon,Horizon,float64,DOUBLE,False,False
3,indicator,Indicator,object,VARCHAR,True,False
4,kind,Kind,object,VARCHAR,True,False


### Construction des tables de dimensions

In [5]:
# Construction des types de dimensions
dimension_tables = schema_builder.create_dimension_tables(column_labels=labels)
dimension_tables['indicator'].head()

2026-01-22 20:13:39,533 - INFO - Successfully extracted meta-data from column 'indicator'
2026-01-22 20:13:39,599 - INFO - The column 'indicator' is of type 'object' and the number of modalities 2 satisfies the categorical threshold criteria 200
2026-01-22 20:13:39,602 - INFO - Successfully extracted meta-data from column 'country'
2026-01-22 20:13:39,654 - INFO - The column 'country' is of type 'object' and the number of modalities 6 satisfies the categorical threshold criteria 200
2026-01-22 20:13:39,656 - INFO - Successfully extracted meta-data from column 'date'
2026-01-22 20:13:39,657 - INFO - Successfully extracted meta-data from column 'value'
2026-01-22 20:13:39,660 - INFO - Successfully extracted meta-data from column 'kind'
2026-01-22 20:13:39,713 - INFO - The column 'kind' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 200
2026-01-22 20:13:39,715 - INFO - Successfully extracted meta-data from column 'horizon'
2026-01-22 20:13:

,value,label
0,0,Gross Domestic Product
1,1,Private Consumption


### Construction de la table d'information

In [6]:
# Construction de la table d'informations
df_fact = schema_builder.create_fact_table(column_labels=labels)
df_fact.head()

C:\Users\bolli\OneDrive\Documents\COD Code\dashboard-template-database\dashboard_template_database\builders\schema.py:181: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df_fact[column] = self.df_fact[column].replace(dict_label_value)
2026-01-22 20:13:43,472 - INFO - Successfully replace modalities by ids in column 'country'
C:\Users\bolli\OneDrive\Documents\COD Code\dashboard-template-database\dashboard_template_database\builders\schema.py:181: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df_fact[column] = self.df_fact[

,indicator,country,date,value,kind,horizon,week,model,training
0,0,0,1960-04-01,0.375710,0,NaN,NaN,0.0,0.0
1,0,0,1960-07-01,0.748561,0,NaN,NaN,0.0,0.0
2,0,0,1960-10-01,1.185218,0,NaN,NaN,0.0,0.0
3,0,0,1961-01-01,1.608374,0,NaN,NaN,0.0,0.0
4,0,0,1961-04-01,1.600329,0,NaN,NaN,0.0,0.0


### Création de l'ensemble des tables

In [7]:
# Création de l'ensemble des tables du schéma
df_metadata, dimension_tables, df_fact = schema_builder.build(column_labels=labels)

2026-01-22 20:13:47,527 - INFO - Successfully extracted meta-data from column 'indicator'
2026-01-22 20:13:47,606 - INFO - The column 'indicator' is of type 'object' and the number of modalities 2 satisfies the categorical threshold criteria 200
2026-01-22 20:13:47,609 - INFO - Successfully extracted meta-data from column 'country'
2026-01-22 20:13:47,673 - INFO - The column 'country' is of type 'object' and the number of modalities 6 satisfies the categorical threshold criteria 200
2026-01-22 20:13:47,675 - INFO - Successfully extracted meta-data from column 'date'
2026-01-22 20:13:47,678 - INFO - Successfully extracted meta-data from column 'value'
2026-01-22 20:13:47,679 - INFO - Successfully extracted meta-data from column 'kind'
2026-01-22 20:13:47,735 - INFO - The column 'kind' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 200
2026-01-22 20:13:47,736 - INFO - Successfully extracted meta-data from column 'horizon'
2026-01-22 20:13:

## 2 - Construction de la base de données

### Initialisation du builder

In [8]:
# Initialisation du builder
builder = DuckdbTablesBuilder(df=df_origin, categorical_threshold=config['THRESHOLD'], path=os.path.join('../', config['OUTPUT_DATA']))

### Création du schéma

In [9]:
# Construction du schéma duckDB
builder.build_duckdb_schema()

2026-01-22 20:13:56,376 - INFO - Successfully extracted meta-data from column 'indicator'
2026-01-22 20:13:56,463 - INFO - The column 'indicator' is of type 'object' and the number of modalities 2 satisfies the categorical threshold criteria 200
2026-01-22 20:13:56,466 - INFO - Successfully extracted meta-data from column 'country'
2026-01-22 20:13:56,528 - INFO - The column 'country' is of type 'object' and the number of modalities 6 satisfies the categorical threshold criteria 200
2026-01-22 20:13:56,531 - INFO - Successfully extracted meta-data from column 'date'
2026-01-22 20:13:56,534 - INFO - Successfully extracted meta-data from column 'value'
2026-01-22 20:13:56,536 - INFO - Successfully extracted meta-data from column 'kind'
2026-01-22 20:13:56,601 - INFO - The column 'kind' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 200
2026-01-22 20:13:56,604 - INFO - Successfully extracted meta-data from column 'horizon'
2026-01-22 20:13:

### Affichage du schéma

In [11]:
# Affichage du schéma
builder.display_schema()

2026-01-22 20:14:06,049 - INFO - 
 Created Tables:
2026-01-22 20:14:06,051 - INFO - 
 dim_country Structure:
2026-01-22 20:14:06,056 - INFO -   value: VARCHAR
2026-01-22 20:14:06,058 - INFO -   label: VARCHAR
2026-01-22 20:14:06,059 - INFO - 
 dim_indicator Structure:
2026-01-22 20:14:06,062 - INFO -   value: VARCHAR
2026-01-22 20:14:06,064 - INFO -   label: VARCHAR
2026-01-22 20:14:06,065 - INFO - 
 dim_kind Structure:
2026-01-22 20:14:06,068 - INFO -   value: VARCHAR
2026-01-22 20:14:06,070 - INFO -   label: VARCHAR
2026-01-22 20:14:06,071 - INFO - 
 dim_model Structure:
2026-01-22 20:14:06,074 - INFO -   value: VARCHAR
2026-01-22 20:14:06,076 - INFO -   label: VARCHAR
2026-01-22 20:14:06,078 - INFO - 
 dim_training Structure:
2026-01-22 20:14:06,080 - INFO -   value: VARCHAR
2026-01-22 20:14:06,082 - INFO -   label: VARCHAR
2026-01-22 20:14:06,085 - INFO - 
 fact_table Structure:
2026-01-22 20:14:06,089 - INFO -   indicator: BIGINT
2026-01-22 20:14:06,091 - INFO -   country: BIGINT


### Exemple de requête

In [12]:
# Requête de la table d'information
print(builder.conn.execute("SELECT * FROM dim_model").fetchall())

[('1', 'ElasticNetCV'), ('2', 'ExtraTrees'), ('3', 'LassoCV'), ('4', 'RandomForest'), ('5', 'RidgeCV'), ('6', 'XGBStandard'), ('0', None)]


## A AJOUTER 
- UN CAS DE CONSTRUCTION  SANS TABLE DE DIMENSION
- UN CAS AVEC DES DONNEES PARQUET, INDEXATION ET PARTITIONNEMENT
